In [1]:
import torch
import cv2
import os
from PIL import Image
import pandas as pd
import numpy as np
import subprocess
from tqdm import tqdm

In [2]:
def _isArrayLike(obj):

    return not type(obj) == str and hasattr(obj, '__iter__') and hasattr(obj, '__len__')

In [3]:
class Databuilder:
    def __init__(self, img_file, save_root, width, height, load_model=True):
        self.img = img_file
        self.save_root = save_root
        self.crop_w = width/2
        self.crop_h = height/2
        self.setup(load_model)

    def setup(self, load_model):
    
        self.crop_save = os.path.join(
            self.save_root, "annotation", "crop"
        )
    
        self.keypoints_save = os.path.join(
            self.save_root, "annotation", "keypoints"
        )
    
        self.img_save = os.path.join(
            self.save_root, "crop_img"
        )
    
        self.keypoints_vis = os.path.join(
            self.save_root, "keypoint_img"
        )
    
        os.makedirs(self.crop_save, exist_ok=True)
        os.makedirs(self.keypoints_save, exist_ok=True)
        os.makedirs(self.img_save, exist_ok=True)
        os.makedirs(self.keypoints_vis, exist_ok=True)
    
        if load_model:
            model = torch.hub.load(
                'ultralytics/yolov5',
                'yolov5x',
                trust_repo=True,
                skip_validation=True
            )
    
            model.conf = 0.01
            model.iou = 0.45
            model.classes = [19]
            model.multi_label = False
    
            self.model = model

    def bbox_crop(self, img_dir):
        img_list = [os.path.join(img_dir, img) for img in os.listdir(
            img_dir) if img.endswith(".jpg")]
        dog_name = img_dir.split("\\")[-1]

        bbox = pd.DataFrame(
            columns=["xmin", "ymin", "xmax", "ymax", "confidence", "class", "name"])
        empty_name = []

        for i, img in enumerate(tqdm(img_list)):
            results = self.model(img)
            df_bb = results.pandas().xyxy[0]
            if df_bb.empty:
                empty_name.append(i)
            else:

                bbox = pd.concat([
                    bbox,
                    df_bb.loc[[df_bb["confidence"].idxmax()]]
                ], ignore_index=True)
        order_list = np.arange(len(img_list))
        order_list = np.delete(order_list, empty_name)
        order_list += 1
        img_list = np.asarray(img_list)
        img_list = np.delete(img_list, empty_name)
        
        img_name = list(
            map(
                lambda item: 
                     item.split("\\")[-1], img_list
            )
        )
        
        im = Image.open(img_list[0])
        img_w, img_h = im.size
        bbox.index = img_name
        
        bboxcrop = self.table_process(bbox, img_w, img_h)
        bboxcrop["order"] = order_list
        return bboxcrop

    def table_process(self, bbox, img_w, img_h):
        # crop 坐标
        bbox_coord = bbox.iloc[:, :4]
        bbox_coord["xcrop"] = (
            (bbox_coord["xmin"]+bbox_coord["xmax"])/2)-self.crop_w
        bbox_coord["ycrop"] = (
            (bbox_coord["ymin"]+bbox_coord["ymax"])/2)-self.crop_h
        bbox_coord["xcrop"] = np.clip(
            bbox_coord["xcrop"], 0, img_w-self.crop_w*2)
        bbox_coord["ycrop"] = np.clip(
            bbox_coord["ycrop"], 0, img_h-self.crop_h*2)

        out_dict = {
            "x": (bbox_coord["xmin"]-bbox_coord["xcrop"]).values,
            "y": (bbox_coord["ymin"]-bbox_coord["ycrop"]).values,
            "w": (bbox_coord["xmax"]-bbox_coord["xmin"]).values,
            "h": (bbox_coord["ymax"]-bbox_coord["ymin"]).values,

        }
        # bbox坐标扩大10%
        out_dict_extend = {
            "x": np.clip(out_dict["x"]-out_dict["w"]*0.1/2, 0, self.crop_w*2),
            "y": np.clip(out_dict["y"]-out_dict["h"]*0.1/2, 0, self.crop_h*2),
            "w": np.clip(out_dict["w"]*1.1, 0, self.crop_w*2),
            "h": np.clip(out_dict["h"]*1.1, 0, self.crop_h*2),
        }
        out_inf_extend = pd.DataFrame(out_dict_extend)
        out_inf_extend.index = bbox_coord.index
        # 合成一个表
        result = pd.concat([bbox_coord, out_inf_extend],
                           axis=1)
        result = result.apply(lambda item: round(item), axis=0)
        return result

    def save_ann(self, bboxcrop, name):
        hdf_file = os.path.join(self.crop_save, name+".h5")
        csv_file = os.path.join(self.crop_save, name+".csv")
        if not os.path.isfile(hdf_file):
            bboxcrop.to_hdf(
                hdf_file, "df_with_missing", format="table", mode="w"
            )
            bboxcrop.to_csv(csv_file)
            

    def save_img(self, crop_df, input_dir, out_dir):
    
        os.makedirs(out_dir, exist_ok=True)
    
        for i, name in enumerate(crop_df.index):
    
            img_path = os.path.join(input_dir, name)
    
            with Image.open(img_path) as img:
    
                img = img.convert("RGB")
    
                x = crop_df["xcrop"].iloc[i]
                y = crop_df["ycrop"].iloc[i]
    
                img = img.crop((
                    x,
                    y,
                    x + self.crop_w * 2,
                    y + self.crop_h * 2
                ))
    
                output_path = os.path.join(out_dir, name)
                img.save(output_path)
    
                print(f"Saved: {output_path}")

    def run_pose(self, hdf_file, crop_img, pose_vis):
        if os.path.isdir(crop_img) and os.path.isfile(hdf_file):
            command = [
                "python",
                "demo/top_down_img_custom.py ",
                "configs/animal/2d_kpt_sview_rgb_img/topdown_heatmap/animalpose/hrnet_w32_animalpose_256x256.py",
                "checkpoints/hrnet_w32_animalpose_256x256-1aa7f075_20210426.pth",
                "--img-root",
                crop_img,
                "--hdf-file",
                hdf_file,
                "--out-df-root",
                self.keypoints_save,
            ]
            if pose_vis:
                name = crop_img.split("\\")[-1]
                pose_img = os.path.join(self.keypoints_vis, name)
                if not os.path.isdir(pose_img):
                    os.makedirs(pose_img)
                command += ["--out-img-root", pose_img]
            subprocess.run(command)

    def pipeline(self, needcrop=True, pose=True, pose_vis=True):
        images = self.img if _isArrayLike(self.img) else [self.img]
        cur_dir = os.getcwd()
        for img_dir in images:
            name = img_dir.split("\\")[-1]
            crop_img = os.path.join(self.img_save, name)
            if needcrop:
                if os.path.isdir(crop_img):
                    continue
                bboxcrop = self.bbox_crop(img_dir)
                
                self.save_ann(bboxcrop, name)
                self.save_img(bboxcrop, img_dir, crop_img)

            if pose:
                hdf_file = os.path.join(self.crop_save, name+".h5")
                pose_hdf = os.path.join(self.keypoints_save, name+".h5")
                if os.path.isfile(pose_hdf):
                    continue
                if os.path.isfile(hdf_file):
                    mmpose_dir = r"D:\pose\mmpose\mmpose"
                    os.chdir(mmpose_dir)
                    self.run_pose(hdf_file, crop_img, pose_vis)
                    os.chdir(cur_dir)



In [12]:
import cv2
import os

def video_to_frames(video_path, output_folder="test_images", target_fps=15):
    os.makedirs(output_folder, exist_ok=True)

    cap = cv2.VideoCapture(video_path)

    if not cap.isOpened():
        raise ValueError(f"Could not open video: {video_path}")

    original_fps = cap.get(cv2.CAP_PROP_FPS)

    if original_fps <= 0:
        raise ValueError("Could not determine video FPS")

    print(f"Original FPS: {original_fps}")
    print(f"Target FPS: {target_fps}")

    frame_number = 0
    saved_number = 0

    while True:
        ret, frame = cap.read()

        if not ret:
            break

        # Save frames at 15 FPS
        current_time = frame_number / original_fps
        expected_saved_frames = int(current_time * target_fps) + 1

        if saved_number < expected_saved_frames:
            frame_name = f"{saved_number:06d}.jpg"
            frame_path = os.path.join(output_folder, frame_name)

            cv2.imwrite(frame_path, frame)
            saved_number += 1

        frame_number += 1

    cap.release()

    print(f"Done! Saved {saved_number} frames at {target_fps} FPS.")

if True:
    video_path = r"C:\Users\sol77\Desktop\CattleLameness\Data\Normal\N (8).mp4"
    output_folder = r"C:\Users\sol77\Desktop\test_images"
    
    video_to_frames(video_path, output_folder, target_fps=10)

Original FPS: 25.0
Target FPS: 10
Done! Saved 55 frames at 10 FPS.


In [13]:
# img_file = r"C:\Users\sol77\Desktop\test.jpg"

# img = ["Mafi-oft-ilk-cam1_Trim"]
# #img_file_list = [os.path.join(img_file, item) for item in os.listdir(img_file)]
# img_file_list = [os.path.join(img_file, item) for item in img]
img_file_list = [r"C:\Users\sol77\Desktop\test_images"]

save_file = r"C:\Users\sol77\Desktop\outputs"

data_model = Databuilder(
    img_file_list,
    save_file,
    800,
    800,
    load_model=True
)

data_model.pipeline(needcrop=True, pose=True, pose_vis=True)

YOLOv5  2026-11-14 Python-3.11.11 torch-2.10.0+cpu CPU



Using cache found in C:\Users\sol77/.cache\torch\hub\ultralytics_yolov5_master


Fusing layers... 
YOLOv5x summary: 256 layers, 86,705,005 parameters, 0 gradients, 205.4 GFLOPs
Adding AutoShape... 


  0%|                                                                                           | 0/55 [00:00<?, ?it/s]C:\Users\sol77\AppData\Local\Temp\ipykernel_13308\3337449786.py:63: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  bbox = pd.concat([
100%|██████████████████████████████████████████████████████████████████████████████████| 55/55 [02:58<00:00,  3.24s/it]
C:\Users\sol77\AppData\Local\Temp\ipykernel_13308\3337449786.py:126: FutureWarning: Starting with pandas version 3.0 all arguments of to_hdf except for the argument 'path_or_buf' will be keyword-only.
  bboxcrop.to_hdf(


Saved: C:\Users\sol77\Desktop\outputs\crop_img\test_images\000000.jpg
Saved: C:\Users\sol77\Desktop\outputs\crop_img\test_images\000001.jpg
Saved: C:\Users\sol77\Desktop\outputs\crop_img\test_images\000002.jpg
Saved: C:\Users\sol77\Desktop\outputs\crop_img\test_images\000003.jpg
Saved: C:\Users\sol77\Desktop\outputs\crop_img\test_images\000004.jpg
Saved: C:\Users\sol77\Desktop\outputs\crop_img\test_images\000005.jpg
Saved: C:\Users\sol77\Desktop\outputs\crop_img\test_images\000006.jpg
Saved: C:\Users\sol77\Desktop\outputs\crop_img\test_images\000007.jpg
Saved: C:\Users\sol77\Desktop\outputs\crop_img\test_images\000008.jpg
Saved: C:\Users\sol77\Desktop\outputs\crop_img\test_images\000009.jpg
Saved: C:\Users\sol77\Desktop\outputs\crop_img\test_images\000010.jpg
Saved: C:\Users\sol77\Desktop\outputs\crop_img\test_images\000011.jpg
Saved: C:\Users\sol77\Desktop\outputs\crop_img\test_images\000012.jpg
Saved: C:\Users\sol77\Desktop\outputs\crop_img\test_images\000013.jpg
Saved: C:\Users\sol7

FileNotFoundError: [WinError 3] The system cannot find the path specified: 'D:\\pose\\mmpose\\mmpose'